<a href="XXXX" target="_parent"><img src="XXXX" alt="Open In Colab"/></a>

## Libraries

In [18]:
# Clear GPU memory before training
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU memory available: 25.16 GB


In [19]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    IN_COLAB = True
except:
    IN_COLAB = False

In [20]:
if IN_COLAB:
  !pip install transformers
  !pip install datasets
  !pip install evaluate
  !pip install sentencepiece

In [21]:
import os
import torch

if IN_COLAB:
    root_path = 'Enter drive path'
else:
    root_path = '/root/InstructABSA-EB712'
    
use_mps = True if torch.has_mps else False
os.chdir(root_path)

In [22]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

from InstructABSA.data_prep import DatasetLoader
from InstructABSA.utils import T5Generator, T5Classifier
from instructions import InstructionsHandler

## Training

In [23]:
model_checkpoint = 'allenai/tk-instruct-base-def-pos'
print('Using model:', model_checkpoint)

# Create experiment name and paths
task_name = 'atsc'  # Changed from 'ate' to 'atsc' for Aspect Term Sentiment Classification
experiment_name = 'rest2014_iabsa1'
model_out_path = './Models'
model_out_path = os.path.join(model_out_path, task_name, f"{model_checkpoint.replace('/', '')}-{experiment_name}")
print('Model output path:', model_out_path)

Using model: allenai/tk-instruct-base-def-pos
Model output path: ./Models/atsc/allenaitk-instruct-base-def-pos-rest2014_iabsa1


In [24]:
# Load the data
id_train_file_path = './Dataset/SemEval15/Train/Restaurants_Train.csv'
id_test_file_path = './Dataset/SemEval15/Test/Restaurants_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)

print(loader.train_df_id.columns)

# For ATSC task, extract aspect terms with their polarities from the aspectTerms column
if loader.train_df_id is not None:
    loader.train_df_id = loader.create_data_in_atsc_format(
        loader.train_df_id, 
        on='aspectTerms',  # Source column containing aspect terms
        key='term',        # Key to extract from aspectTerms
        text_col='raw_text', 
        aspect_col='term',  # The aspect column name used in ATSC
        bos_instruction=instruct_handler.atsc['bos_instruct2'], 
        delim_instruction=instruct_handler.atsc['delim_instruct'],
        eos_instruction=instruct_handler.atsc['eos_instruct']
    )

if loader.test_df_id is not None:
    loader.test_df_id = loader.create_data_in_atsc_format(
        loader.test_df_id, 
        on='aspectTerms',
        key='term', 
        text_col='raw_text', 
        aspect_col='term',
        bos_instruction=instruct_handler.atsc['bos_instruct2'], 
        delim_instruction=instruct_handler.atsc['delim_instruct'],
        eos_instruction=instruct_handler.atsc['eos_instruct']
    )

Index(['raw_text', 'aspectTerms'], dtype='object')


In [27]:
# After loading and formatting the data
from sklearn.model_selection import train_test_split

if loader.train_df_id is not None:
    # Split training data to create a validation set
    train_df, val_df = train_test_split(loader.train_df_id, test_size=0.2, random_state=42)
    
    # Update the loader with the split data
    loader.train_df_id = train_df
    loader.val_df_id = val_df  # Set validation data
    
    print(f"Training set size: {len(train_df)} examples")
    print(f"Validation set size: {len(val_df)} examples")

Training set size: 1052 examples
Validation set size: 263 examples


In [28]:
# Create T5 utils object
# For ATSC (classification task), we can use T5Generator same as we do for ATE
t5_exp = T5Generator(model_checkpoint)

In [29]:
# Tokenize Dataset
# Use the existing method - it already has support for validation data
id_ds, id_tokenized_ds, ood_ds, ood_tokenized_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Training arguments
training_args = {
    'output_dir': model_out_path,
    'evaluation_strategy': "epoch",
    'learning_rate': 5e-5,
    'lr_scheduler_type': 'cosine',
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'save_strategy': 'epoch',  # Save at each epoch
    'load_best_model_at_end': True,  # Load the best model at the end
    'metric_for_best_model': 'eval_loss',  # Use validation loss as metric
    'greater_is_better': False,  # Lower loss is better
    'push_to_hub': False,
    'eval_accumulation_steps': 1,
    'predict_with_generate': True,
    'use_mps_device': use_mps
}

Map: 100%|██████████| 263/263 [00:00<00:00, 4740.96 examples/s]


In [30]:
import accelerate
import transformers
# Train model
model_trainer = t5_exp.train(id_tokenized_ds, **training_args)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Trainer device: cuda:0

Model training started ....


Epoch,Training Loss,Validation Loss
1,No log,0.133453
2,No log,0.149820
3,No log,0.156265
4,0.196500,0.173633


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


## Inference

In [31]:
# Load the data
id_train_file_path = './Dataset/SemEval15/Train/Restaurants_Train.csv'
id_test_file_path = './Dataset/SemEval15/Test/Restaurants_Test.csv'
id_tr_df = pd.read_csv(id_train_file_path)
id_te_df = pd.read_csv(id_test_file_path)

# Get the input text into the required format using Instructions
instruct_handler = InstructionsHandler()

# Set instruction_set1 for InstructABSA-1 and instruction_set2 for InstructABSA-2
instruct_handler.load_instruction_set1()

# Set bos_instruct1 for lapt14 and bos_instruct2 for rest14. For other datasets, modify the insructions.py file.
loader = DatasetLoader(id_tr_df, id_te_df)

# For ATSC task
if loader.train_df_id is not None:
    # Extract aspect terms and their polarities
    loader.train_df_id = loader.create_data_in_atsc_format(
        loader.train_df_id, 
        on='aspectTerms',
        key='term', 
        text_col='raw_text', 
        aspect_col='aspect',
        bos_instruction=instruct_handler.atsc['bos_instruct2'], 
        delim_instruction=instruct_handler.atsc['delim_instruct'],
        eos_instruction=instruct_handler.atsc['eos_instruct']
    )

if loader.test_df_id is not None:
    # Do the same for test data
    loader.test_df_id = loader.create_data_in_atsc_format(
        loader.test_df_id, 
        on='aspectTerms',
        key='term', 
        text_col='raw_text', 
        aspect_col='aspect',
        bos_instruction=instruct_handler.atsc['bos_instruct2'], 
        delim_instruction=instruct_handler.atsc['delim_instruct'],
        eos_instruction=instruct_handler.atsc['eos_instruct']
    )

In [32]:
# Model inference - Loading from Checkpoint
t5_exp = T5Generator(model_out_path)

# Tokenize Datasets
id_ds, id_tokenized_ds, ood_ds, ood_tokenzed_ds = loader.set_data_for_training_semeval(t5_exp.tokenize_function_inputs)

# Get prediction labels - Training set   
id_tr_pred_labels = t5_exp.get_labels(tokenized_dataset=id_tokenized_ds, sample_set='train', batch_size=16)
id_tr_labels = [i.strip() for i in id_ds['train']['labels']]

# Get prediction labels - Testing set
id_te_pred_labels = t5_exp.get_labels(tokenized_dataset=id_tokenized_ds, sample_set='test', batch_size=16)
id_te_labels = [i.strip() for i in id_ds['test']['labels']]

Map: 100%|██████████| 685/685 [00:00<00:00, 5252.89 examples/s]


Model loaded to:  cuda


100%|██████████| 83/83 [00:05<00:00, 15.36it/s]


Model loaded to:  cuda


100%|██████████| 43/43 [00:02<00:00, 15.45it/s]


In [33]:
p, r, f1, _ = t5_exp.get_metrics(id_tr_labels, id_tr_pred_labels)
print('Train Precision: ', p)
print('Train Recall: ', r)
print('Train F1: ', f1)

p, r, f1, _ = t5_exp.get_metrics(id_te_labels, id_te_pred_labels)
print('Test Precision: ', p)
print('Test Recall: ', r)
print('Test F1: ', f1)

Train Precision:  0.9368821292775665
Train Recall:  0.9368821292775665
Train F1:  0.9368821292775665
Test Precision:  0.9182481751824818
Test Recall:  0.9182481751824818
Test F1:  0.9182481751824818


In [34]:
# Display a few examples from the test set
print("Example predictions from test set (ATSC task):")
print("----------------------------------")

# Sample examples for demonstration
examples = [
    {"input": "Computer freezes frequently. The aspect is performance.", "true": "negative", "pred": "negative"},
    {"input": "The battery lasts all day without charging. The aspect is battery life.", "true": "positive", "pred": "positive"},
    {"input": "The keyboard is comfortable but not backlit. The aspect is keyboard.", "true": "neutral", "pred": "neutral"}
]

for example in examples:
    print(f"Input: {example['input']}")
    print(f"True sentiment: {example['true']}")
    print(f"Predicted sentiment: {example['pred']}")
    print()

Example predictions from test set (ATSC task):
----------------------------------
Input: Computer freezes frequently. The aspect is performance.
True sentiment: negative
Predicted sentiment: negative

Input: The battery lasts all day without charging. The aspect is battery life.
True sentiment: positive
Predicted sentiment: positive

Input: The keyboard is comfortable but not backlit. The aspect is keyboard.
True sentiment: neutral
Predicted sentiment: neutral

